# Experimento con modelos alternativos y parámetros fijos

Notebook complementaria a `02_experimento_params_fijos.ipynb`.

Objetivo: evaluar PC-SMOTE y técnicas comparativas con clasificadores usados en papers de referencia, manteniendo hiperparámetros fijos.

Referencias operativas usadas:

- Malyarchuk, Shuttle: XGBoost como modelo fuerte para Shuttle. El paper indica uso de `xgboost`, profundidad default 6 y learning rate default 0.3; se usa `n_estimators=100`, default de la interfaz sklearn de XGBoost.
- Sonoda et al., Communities and Crime / US Crime: Logistic Regression, SVM y Naive Bayes implementados en scikit-learn con parámetros default. Se mantiene esa idea y se fija reproducibilidad cuando aplica.

Alcance por defecto:

- `shuttle`: XGBoost.
- `us_crime`: XGBoost, SVM, Naive Bayes y Logistic Regression.

Métricas agregadas:

- `F1_macro`
- `F1_weighted`
- `Balanced Accuracy`
- `AvAcc` (equivalente a Balanced Accuracy: promedio de recalls por clase; en binario coincide con (TPR + TNR) / 2)
- `Recall_macro`
- `Accuracy`
- tiempos de carga, entrenamiento, predicción y total.


In [1]:
import sys
sys.path.append("../scripts")
sys.path.append("../datasets")

import os

# Rutas de datasets y resultados
RUTA_DATASETS_BASE = "../datasets/datasets_aumentados/base/"
RUTA_DATASETS_AUMENTADOS = "../datasets/datasets_aumentados/"
RUTA_DATASETS_CLASICOS = "../datasets/datasets_aumentados/resampler_clasicos/"
RUTA_DATASETS_CONTEMPORANEOS = "../datasets/datasets_aumentados/contemporaneos/"
DIRECTORIO_SALIDA = "../resultados"

os.makedirs(DIRECTORIO_SALIDA, exist_ok=True)
os.makedirs(RUTA_DATASETS_CLASICOS, exist_ok=True)
os.makedirs(RUTA_DATASETS_CONTEMPORANEOS, exist_ok=True)

In [2]:
import gc, time
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd

from sklearn.metrics import (
    f1_score,
    balanced_accuracy_score,
    recall_score,
    accuracy_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)

from xgboost import XGBClassifier

RANDOM_STATE = 42

# =========================================================
# HIPERPARÁMETROS FIJOS POR MODELO
# =========================================================
# Malyarchuk: XGBoost con defaults clave: max_depth=6, learning_rate=0.3.
# Sonoda: LR, SVM y NB de scikit-learn con defaults. Se fija random_state donde aplica.
MODELOS = {
    "XGBoost": {
        "constructor": XGBClassifier,
        "params": {
            "n_estimators": 100,
            "max_depth": 6,
            "learning_rate": 0.3,
            # objective/eval_metric se define dinámicamente según cantidad de clases.
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": 1,
            "verbosity": 0,
        },
        "referencia": "Malyarchuk: xgboost, defaults max_depth=6 y learning_rate=0.3; n_estimators=100 default sklearn API",
    },
    "SVM": {
        "constructor": SVC,
        "params": {
            "C": 1.0,
            "kernel": "rbf",
            "gamma": "scale",
        },
        "referencia": "Sonoda: SVM scikit-learn con parámetros default",
    },
    "NaiveBayes": {
        "constructor": GaussianNB,
        "params": {},
        "referencia": "Sonoda: Naive Bayes scikit-learn con parámetros default; GaussianNB para atributos numéricos",
    },
    "LogisticRegression": {
        "constructor": LogisticRegression,
        "params": {
            "penalty": "l2",
            "C": 1.0,
            "solver": "lbfgs",
            "max_iter": 1000,
            "random_state": RANDOM_STATE,
        },
        "referencia": "Sonoda: Logistic Regression scikit-learn default; max_iter=1000 fijo para asegurar convergencia",
    },
}

MODELOS_POR_DATASET = {
    "shuttle": ["XGBoost"],
    "us_crime": ["XGBoost", "SVM", "NaiveBayes", "LogisticRegression"],
}

NOMBRE_ARCHIVO_EXCEL = os.path.join(DIRECTORIO_SALIDA, "resultados_modelos_alternativos_params_fijos.xlsx")

print("Modelos fijos cargados:")
for nombre, spec in MODELOS.items():
    print(f"- {nombre}: {spec['params']}")


Modelos fijos cargados:
- XGBoost: {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.3, 'tree_method': 'hist', 'random_state': 42, 'n_jobs': 1, 'verbosity': 0}
- SVM: {'C': 1.0, 'kernel': 'rbf', 'gamma': 'scale'}
- NaiveBayes: {}
- LogisticRegression: {'penalty': 'l2', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 1000, 'random_state': 42}


In [3]:
from pathlib import Path
import re

@dataclass
class DatasetCombination:
    dataset_logico: str
    tipo_combination: str      # "base" | "clasico" | "contemporaneo" | "pcsmote"
    ruta_train_csv: str
    ruta_test_csv: str

    tecnica_aumento: str = "base"

    valor_densidad: str | int | None = "--"
    valor_riesgo: str | int | None = "--"
    criterio_pureza: str | None = "--"

    percentil_radio_distancia: str | int | None = "--"
    percentil_riesgo: str | int | None = "--"
    umbral_densidad: str | None = "--"
    umbral_riesgo: str | None = "--"

    tipo_pureza: str = "--"
    nombre_configuracion: str = ""

    grado_limpieza: str | int = "--"
    total_muestras_train: int | None = None
    tamanio_dataset: int | None = None
    sinteticos_generados: int = 0
    semillas_validas: int = 0
    semillas_analizadas: int = 0


@dataclass
class RegistroRendimiento:
    dataset_logico: str
    tipo_combination: str
    nombre_modelo_aprendizaje: str
    nombre_configuracion: str
    tecnica_aumento: str
    valor_densidad: str
    valor_riesgo: str
    criterio_pureza: str
    grado_limpieza: str

    cantidad_train: int
    cantidad_test: int
    cantidad_caracteristicas: int

    test_f1_macro: float
    test_f1_weighted: float
    test_balanced_accuracy: float
    test_avacc: float
    test_recall_macro: float
    test_accuracy: float

    hiperparametros_usados: str
    referencia_configuracion: str
    tiempo_carga_seg: float
    tiempo_entrenamiento_seg: float
    tiempo_prediccion_seg: float
    tiempo_total_seg: float


In [4]:
def enumerar_combinaciones_base_y_aumentadas(
    ruta_base,
    ruta_clasicos,
    ruta_contemporaneos,
    ruta_aumentados,
    verbose=True
):
    combinaciones = []
    cont_combinaciones = 0
    tamanio_train_base_por_dataset_y_I = {}

    if verbose:
        print(f"[SCAN] Explorando carpeta base: {ruta_base}")
    archivos_base = os.listdir(ruta_base)

    def resolver_test(dataset_logico):
        patron_test = re.compile(rf"^{re.escape(dataset_logico)}_tdataset(\d+)_tm(\d+)_test\.csv$")
        nombre_test = None
        n_test_detectado = None
        tamanio_dataset = None

        for nc in archivos_base:
            m_test = patron_test.match(nc)
            if m_test:
                n_tm = int(m_test.group(2))
                if n_test_detectado is None or n_tm > n_test_detectado:
                    n_test_detectado = n_tm
                    nombre_test = nc
                    tamanio_dataset = int(m_test.group(1))

        return nombre_test, tamanio_dataset

    # 1) BASE
    for nombre in archivos_base:
        if not nombre.endswith("_train.csv"):
            continue

        m = re.match(r"(.+?)_I(\d+)_tm(\d+)_train\.csv$", nombre)
        if not m:
            continue

        dataset_logico = m.group(1)
        grado_limpieza = int(m.group(2))
        total_muestras_train = int(m.group(3))
        clave_base = (dataset_logico, grado_limpieza)
        tamanio_train_base_por_dataset_y_I[clave_base] = total_muestras_train

        nombre_test, tamanio_dataset = resolver_test(dataset_logico)
        if nombre_test is None:
            continue

        cont_combinaciones += 1
        if verbose:
            print(f"#{cont_combinaciones}  [OK] BASE: {nombre}")

        combinaciones.append(DatasetCombination(
            dataset_logico=dataset_logico,
            tipo_combination="base",
            ruta_train_csv=os.path.join(ruta_base, nombre),
            ruta_test_csv=os.path.join(ruta_base, nombre_test),
            tecnica_aumento="base",
            grado_limpieza=grado_limpieza,
            total_muestras_train=total_muestras_train,
            tamanio_dataset=tamanio_dataset,
        ))

    def agregar_combinaciones_simples(ruta_carpeta, tipo_combination, etiqueta):
        nonlocal cont_combinaciones

        if not os.path.isdir(ruta_carpeta):
            if verbose:
                print(f"[WARN] Carpeta inexistente: {ruta_carpeta}")
            return

        if verbose:
            print(f"[SCAN] Explorando carpeta {etiqueta}: {ruta_carpeta}")

        for nombre in os.listdir(ruta_carpeta):
            if not nombre.endswith("_train.csv"):
                continue

            m = re.match(r"(.+?)_(.+?)_I(\d+)_sg(\d+)_train\.csv$", nombre)
            if not m:
                continue

            tecnica = m.group(1).lower()
            dataset_logico = m.group(2)
            grado_limpieza = int(m.group(3))
            sinteticos_generados = int(m.group(4))

            clave_base = (dataset_logico, grado_limpieza)
            total_muestras_train = tamanio_train_base_por_dataset_y_I.get(clave_base)
            if total_muestras_train is None:
                continue

            nombre_test, tamanio_dataset = resolver_test(dataset_logico)
            if nombre_test is None:
                continue

            cont_combinaciones += 1
            if verbose:
                print(f"#{cont_combinaciones}  [OK] {etiqueta.upper()} ({tecnica}): {nombre}")

            combinaciones.append(DatasetCombination(
                dataset_logico=dataset_logico,
                tipo_combination=tipo_combination,
                ruta_train_csv=os.path.join(ruta_carpeta, nombre),
                ruta_test_csv=os.path.join(ruta_base, nombre_test),
                tecnica_aumento=tecnica,
                grado_limpieza=grado_limpieza,
                total_muestras_train=total_muestras_train,
                sinteticos_generados=sinteticos_generados,
                tamanio_dataset=tamanio_dataset,
            ))

    # 2) CLÁSICOS
    agregar_combinaciones_simples(
        ruta_carpeta=ruta_clasicos,
        tipo_combination="clasico",
        etiqueta="clásicos",
    )

    # 3) CONTEMPORÁNEOS
    agregar_combinaciones_simples(
        ruta_carpeta=ruta_contemporaneos,
        tipo_combination="contemporaneo",
        etiqueta="contemporáneos",
    )

    # 4) PC-SMOTE
    if verbose:
        print(f"[SCAN] Explorando carpeta aumentados: {ruta_aumentados}")
    archivos_aumentados = os.listdir(ruta_aumentados)
    patron_pcsmote = re.compile(
        r"^pcs_(?P<dataset>.+?)_"
        r"PRD(?P<prd>\d+)_"
        r"PR(?P<pr>\d+)_"
        r"CP(?P<cp>(?:ent|prop))_"
        r"UD(?P<ud>\d{3})_"
        r"(?P<tipo_pureza>(?:PE\d+|Upp\d{3}))_"
        r"UR(?P<ur>\d{3})_"
        r"I(?P<iso>\d+)_"
        r"SC(?P<sc>\d+)_"
        r"SV(?P<sv>\d+)_"
        r"SG(?P<sg>\d+)_train\.csv$"
    )

    for nombre in archivos_aumentados:
        if not nombre.endswith("_train.csv"):
            continue

        m = patron_pcsmote.match(nombre)
        if not m:
            continue

        dataset_logico = m.group("dataset")
        valor_densidad = int(m.group("prd"))
        valor_riesgo = int(m.group("pr"))
        cp_code = m.group("cp")
        ud_str = m.group("ud")
        ur_str = m.group("ur")
        tipo_pureza = m.group("tipo_pureza")
        grado_limpieza = int(m.group("iso"))
        semillas_analizadas = int(m.group("sc"))
        semillas_validas = int(m.group("sv"))
        sinteticos_generados = int(m.group("sg"))
        criterio_pureza = "entropia" if cp_code == "ent" else "proporcion"

        nombre_configuracion = (
            f"PRD{valor_densidad}_PR{valor_riesgo}_CP{cp_code}_"
            f"UD{ud_str}_UR{ur_str}_{tipo_pureza}_I{grado_limpieza}_"
            f"SC{semillas_analizadas}_SV{semillas_validas}_SG{sinteticos_generados}"
        )

        nombre_test, tamanio_dataset = resolver_test(dataset_logico)
        if nombre_test is None:
            continue

        cont_combinaciones += 1
        if verbose:
            print(f"#{cont_combinaciones}  [OK] PC-SMOTE: {nombre}")

        combinaciones.append(DatasetCombination(
            dataset_logico=dataset_logico,
            tipo_combination="pcsmote",
            ruta_train_csv=os.path.join(ruta_aumentados, nombre),
            ruta_test_csv=os.path.join(ruta_base, nombre_test),
            tecnica_aumento="pcsmote",
            valor_densidad=valor_densidad,
            valor_riesgo=valor_riesgo,
            criterio_pureza=criterio_pureza,
            percentil_radio_distancia=valor_densidad,
            percentil_riesgo=valor_riesgo,
            umbral_densidad=ud_str,
            umbral_riesgo=ur_str,
            grado_limpieza=grado_limpieza,
            sinteticos_generados=sinteticos_generados,
            semillas_validas=semillas_validas,
            semillas_analizadas=semillas_analizadas,
            tipo_pureza=tipo_pureza,
            nombre_configuracion=nombre_configuracion,
            tamanio_dataset=tamanio_dataset,
        ))

    if verbose:
        print(f"\n[INFO] Total combinaciones: {len(combinaciones)}")
    return combinaciones


print("[INFO] Enumerando combinaciones...")
combinaciones = enumerar_combinaciones_base_y_aumentadas(
    ruta_base=RUTA_DATASETS_BASE,
    ruta_clasicos=RUTA_DATASETS_CLASICOS,
    ruta_contemporaneos=RUTA_DATASETS_CONTEMPORANEOS,
    ruta_aumentados=RUTA_DATASETS_AUMENTADOS,
    verbose=True
)

[INFO] Enumerando combinaciones...
[SCAN] Explorando carpeta base: ../datasets/datasets_aumentados/base/
#1  [OK] BASE: predict_faults_I0_tm8000_train.csv
#2  [OK] BASE: predict_faults_I1_tm7919_train.csv
#3  [OK] BASE: predict_faults_I3_tm7759_train.csv
#4  [OK] BASE: shuttle_I0_tm46400_train.csv
#5  [OK] BASE: shuttle_I1_tm45932_train.csv
#6  [OK] BASE: telco_churn_I0_tm5634_train.csv
#7  [OK] BASE: telco_churn_I1_tm5577_train.csv
#8  [OK] BASE: us_crime_I0_tm1595_train.csv
#9  [OK] BASE: us_crime_I1_tm1578_train.csv
#10  [OK] BASE: us_crime_I3_tm1546_train.csv
[SCAN] Explorando carpeta clásicos: ../datasets/datasets_aumentados/resampler_clasicos/
#11  [OK] CLÁSICOS (adasyn): adasyn_predict_faults_I0_sg7453_train.csv
#12  [OK] CLÁSICOS (adasyn): adasyn_predict_faults_I1_sg7380_train.csv
#13  [OK] CLÁSICOS (adasyn): adasyn_predict_faults_I3_sg7185_train.csv
#14  [OK] CLÁSICOS (adasyn): adasyn_shuttle_I0_sg208961_train.csv
#15  [OK] CLÁSICOS (adasyn): adasyn_shuttle_I1_sg206831_train.c

In [5]:
# Alcance experimental: datasets y modelos definidos por papers de referencia.
DATASETS_OBJETIVO = set(MODELOS_POR_DATASET.keys())

tareas = [
    combo for combo in combinaciones
    if combo.dataset_logico.lower() in DATASETS_OBJETIVO
]

print(f"Total de tareas dataset/técnica: {len(tareas)}")
print(f"Datasets: {sorted(set(c.dataset_logico for c in tareas))}")
for dataset, modelos in MODELOS_POR_DATASET.items():
    print(f"{dataset}: {modelos}")


Total de tareas dataset/técnica: 56
Datasets: ['shuttle', 'us_crime']
shuttle: ['XGBoost']
us_crime: ['XGBoost', 'SVM', 'NaiveBayes', 'LogisticRegression']


In [6]:
# =========================================================
# UTILIDADES
# =========================================================

def cargar_xy(ruta_csv):
    """Lee un CSV y devuelve (X, y). Usa 'target' si existe, si no la ?ltima columna."""
    df = pd.read_csv(ruta_csv)
    if "target" in df.columns:
        X = df.drop(columns=["target"]).to_numpy(dtype=np.float32, copy=False)
        y = df["target"].to_numpy()
    else:
        X = df.iloc[:, :-1].to_numpy(dtype=np.float32, copy=False)
        y = df.iloc[:, -1].to_numpy()
    return X, y


def codificar_y(y_train, y_test):
    """Codifica etiquetas para modelos como XGBoost, preservando correspondencia train/test."""
    le = LabelEncoder()
    le.fit(np.concatenate([y_train, y_test]))
    return le.transform(y_train), le.transform(y_test), list(le.classes_)


def construir_modelo(nombre_modelo, n_clases):
    spec = MODELOS[nombre_modelo]
    params = dict(spec["params"])
    if nombre_modelo == "XGBoost":
        if n_clases <= 2:
            params.update({"objective": "binary:logistic", "eval_metric": "logloss"})
        else:
            params.update({"objective": "multi:softprob", "eval_metric": "mlogloss", "num_class": n_clases})
    return spec["constructor"](**params)


def entrenar_y_evaluar(nombre_modelo, X_train, y_train, X_test, y_test):
    """Entrena y evalúa un modelo fijo. Retorna métricas y tiempos separados."""
    y_train_enc, y_test_enc, clases = codificar_y(y_train, y_test)

    n_clases = len(np.unique(y_train_enc))
    modelo = construir_modelo(nombre_modelo, n_clases=n_clases)

    t0 = time.perf_counter()
    modelo.fit(X_train, y_train_enc)
    t1 = time.perf_counter()
    y_pred = modelo.predict(X_test)
    t2 = time.perf_counter()

    avacc = balanced_accuracy_score(y_test_enc, y_pred)

    return dict(
        tiempo_entrenamiento=float(t1 - t0),
        tiempo_prediccion=float(t2 - t1),
        tiempo_total=float(t2 - t0),
        clases=str(clases),
        f1_macro=float(f1_score(y_test_enc, y_pred, average="macro", zero_division=0)),
        f1_weighted=float(f1_score(y_test_enc, y_pred, average="weighted", zero_division=0)),
        balanced_accuracy=float(avacc),
        avacc=float(avacc),
        recall_macro=float(recall_score(y_test_enc, y_pred, average="macro", zero_division=0)),
        accuracy=float(accuracy_score(y_test_enc, y_pred)),
    )


In [7]:
# =========================================================
# EVALUACIÓN MASIVA - modelos alternativos con hiperparámetros fijos
# =========================================================

registros = []
fallos = []
pares = []
for combo in tareas:
    for nombre_modelo in MODELOS_POR_DATASET.get(combo.dataset_logico.lower(), []):
        pares.append((combo, nombre_modelo))

total = len(pares)
inicio_total = time.perf_counter()

print("="*100)
print("EVALUACIÓN MODELOS ALTERNATIVOS - hiperparámetros fijos")
print(f"Total corridas: {total}")
print("="*100)

for idx, (combo, nombre_modelo) in enumerate(pares, start=1):
    print(f"\n[{idx}/{total}] Dataset={combo.dataset_logico} | Modelo={nombre_modelo} | Tipo={combo.tipo_combination} | Técnica={combo.tecnica_aumento}")
    print(f"Train: {os.path.basename(combo.ruta_train_csv)}")

    try:
        t_carga0 = time.perf_counter()
        X_train, y_train = cargar_xy(combo.ruta_train_csv)
        X_test,  y_test  = cargar_xy(combo.ruta_test_csv)
        tiempo_carga = time.perf_counter() - t_carga0
    except Exception as e:
        print(f"ERROR carga CSV: {e}")
        fallos.append({"idx": idx, "dataset": combo.dataset_logico, "modelo": nombre_modelo, "error": repr(e), "fase": "carga"})
        continue

    try:
        res = entrenar_y_evaluar(
            nombre_modelo=nombre_modelo,
            X_train=X_train, y_train=y_train,
            X_test=X_test,   y_test=y_test,
        )
    except Exception as e:
        print(f"ERROR entrenamiento/evaluación: {e}")
        fallos.append({"idx": idx, "dataset": combo.dataset_logico, "modelo": nombre_modelo, "error": repr(e), "fase": "entrenamiento"})
        continue

    print(f"OK total={res['tiempo_total']:.2f}s | train={res['tiempo_entrenamiento']:.2f}s | pred={res['tiempo_prediccion']:.2f}s")
    print(f"F1M={res['f1_macro']:.4f} | AvAcc={res['avacc']:.4f} | Acc={res['accuracy']:.4f}")

    spec = MODELOS[nombre_modelo]
    registros.append(asdict(RegistroRendimiento(
        dataset_logico=combo.dataset_logico,
        tipo_combination=combo.tipo_combination,
        nombre_modelo_aprendizaje=nombre_modelo,
        nombre_configuracion=combo.nombre_configuracion,
        tecnica_aumento=combo.tecnica_aumento,
        valor_densidad=str(combo.valor_densidad),
        valor_riesgo=str(combo.valor_riesgo),
        criterio_pureza=str(combo.criterio_pureza),
        grado_limpieza=str(combo.grado_limpieza),

        cantidad_train=int(X_train.shape[0]),
        cantidad_test=int(X_test.shape[0]),
        cantidad_caracteristicas=int(X_train.shape[1]),

        test_f1_macro=round(res["f1_macro"], 4),
        test_f1_weighted=round(res["f1_weighted"], 4),
        test_balanced_accuracy=round(res["balanced_accuracy"], 4),
        test_avacc=round(res["avacc"], 4),
        test_recall_macro=round(res["recall_macro"], 4),
        test_accuracy=round(res["accuracy"], 4),

        hiperparametros_usados=str(spec["params"]),
        referencia_configuracion=spec["referencia"],
        tiempo_carga_seg=round(tiempo_carga, 3),
        tiempo_entrenamiento_seg=round(res["tiempo_entrenamiento"], 3),
        tiempo_prediccion_seg=round(res["tiempo_prediccion"], 3),
        tiempo_total_seg=round(res["tiempo_total"], 3),
    )))

    del X_train, y_train, X_test, y_test
    gc.collect()

elapsed_total = time.perf_counter() - inicio_total
print(f"\nTiempo total: {elapsed_total/60:.2f} min")
print(f"Registros generados: {len(registros)}")
print(f"Fallos: {len(fallos)}")


EVALUACIÓN MODELOS ALTERNATIVOS - hiperparámetros fijos
Total corridas: 170

[1/170] Dataset=shuttle | Modelo=XGBoost | Tipo=base | Técnica=base
Train: shuttle_I0_tm46400_train.csv


OK total=1.68s | train=1.65s | pred=0.03s
F1M=0.9501 | AvAcc=0.9243 | Acc=0.9997

[2/170] Dataset=shuttle | Modelo=XGBoost | Tipo=base | Técnica=base
Train: shuttle_I1_tm45932_train.csv


OK total=1.72s | train=1.69s | pred=0.03s
F1M=0.9520 | AvAcc=0.9281 | Acc=0.9997

[3/170] Dataset=us_crime | Modelo=XGBoost | Tipo=base | Técnica=base
Train: us_crime_I0_tm1595_train.csv


OK total=0.21s | train=0.21s | pred=0.00s
F1M=0.7653 | AvAcc=0.7252 | Acc=0.9449

[4/170] Dataset=us_crime | Modelo=SVM | Tipo=base | Técnica=base
Train: us_crime_I0_tm1595_train.csv
OK total=0.05s | train=0.03s | pred=0.02s
F1M=0.5417 | AvAcc=0.5320 | Acc=0.9273

[5/170] Dataset=us_crime | Modelo=NaiveBayes | Tipo=base | Técnica=base
Train: us_crime_I0_tm1595_train.csv
OK total=0.00s | train=0.00s | pred=0.00s
F1M=0.6901 | AvAcc=0.8714 | Acc=0.8471



[6/170] Dataset=us_crime | Modelo=LogisticRegression | Tipo=base | Técnica=base
Train: us_crime_I0_tm1595_train.csv


...


[165/170] Dataset=us_crime | Modelo=NaiveBayes | Tipo=pcsmote | Técnica=pcsmote
Train: pcs_us_crime_PRD90_PR45_CPprop_UD035_Upp060_UR050_I3_SC116_SV006_SG1314_train.csv
OK total=0.00s | train=0.00s | pred=0.00s
F1M=0.7485 | AvAcc=0.8148 | Acc=0.9123

[166/170] Dataset=us_crime | Modelo=LogisticRegression | Tipo=pcsmote | Técnica=pcsmote
Train: pcs_us_crime_PRD90_PR45_CPprop_UD035_Upp060_UR050_I3_SC116_SV006_SG1314_train.csv


OK total=0.28s | train=0.28s | pred=0.00s
F1M=0.7382 | AvAcc=0.7715 | Acc=0.9173

[167/170] Dataset=us_crime | Modelo=XGBoost | Tipo=pcsmote | Técnica=pcsmote
Train: pcs_us_crime_PRD90_PR45_CPprop_UD045_Upp050_UR050_I3_SC116_SV016_SG1314_train.csv


OK total=0.47s | train=0.46s | pred=0.00s
F1M=0.7583 | AvAcc=0.7504 | Acc=0.9348

[168/170] Dataset=us_crime | Modelo=SVM | Tipo=pcsmote | Técnica=pcsmote
Train: pcs_us_crime_PRD90_PR45_CPprop_UD045_Upp050_UR050_I3_SC116_SV016_SG1314_train.csv
OK total=0.10s | train=0.08s | pred=0.02s
F1M=0.7025 | AvAcc=0.7341 | Acc=0.9048



[169/170] Dataset=us_crime | Modelo=NaiveBayes | Tipo=pcsmote | Técnica=pcsmote
Train: pcs_us_crime_PRD90_PR45_CPprop_UD045_Upp050_UR050_I3_SC116_SV016_SG1314_train.csv
OK total=0.00s | train=0.00s | pred=0.00s
F1M=0.6985 | AvAcc=0.8098 | Acc=0.8747

[170/170] Dataset=us_crime | Modelo=LogisticRegression | Tipo=pcsmote | Técnica=pcsmote
Train: pcs_us_crime_PRD90_PR45_CPprop_UD045_Upp050_UR050_I3_SC116_SV016_SG1314_train.csv


OK total=0.31s | train=0.31s | pred=0.00s
F1M=0.7405 | AvAcc=0.7855 | Acc=0.9148

Tiempo total: 3.86 min
Registros generados: 170
Fallos: 0


In [8]:
# =========================================================
# GUARDADO Y DELTAS VS BASELINE DEL MISMO MODELO
# =========================================================

df = pd.DataFrame(registros)
df_fallos = pd.DataFrame(fallos)

if not df.empty:
    baseline = (
        df[df["tipo_combination"] == "base"]
        .set_index(["dataset_logico", "nombre_modelo_aprendizaje", "grado_limpieza"])["test_f1_macro"]
        .to_dict()
    )

    def delta_vs_base(row):
        key = (row["dataset_logico"], row["nombre_modelo_aprendizaje"], row["grado_limpieza"])
        base = baseline.get(key)
        if base is None:
            return np.nan
        return round(row["test_f1_macro"] - base, 4)

    df["delta_f1_macro_vs_base_mismo_modelo"] = df.apply(delta_vs_base, axis=1)

    columnas_orden = [
        "dataset_logico", "nombre_modelo_aprendizaje", "tipo_combination", "tecnica_aumento", "grado_limpieza",
        "cantidad_train", "cantidad_test", "cantidad_caracteristicas",
        "test_f1_macro", "delta_f1_macro_vs_base_mismo_modelo", "test_avacc", "test_balanced_accuracy",
        "test_f1_weighted", "test_recall_macro", "test_accuracy",
        "tiempo_carga_seg", "tiempo_entrenamiento_seg", "tiempo_prediccion_seg", "tiempo_total_seg",
        "nombre_configuracion", "valor_densidad", "valor_riesgo", "criterio_pureza",
        "hiperparametros_usados", "referencia_configuracion",
    ]
    df = df[columnas_orden]

    resumen = (
        df.sort_values(["dataset_logico", "nombre_modelo_aprendizaje", "test_f1_macro"], ascending=[True, True, False])
        .groupby(["dataset_logico", "nombre_modelo_aprendizaje"], as_index=False)
        .head(10)
    )
else:
    resumen = pd.DataFrame()

with pd.ExcelWriter(NOMBRE_ARCHIVO_EXCEL, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="resultados", index=False)
    resumen.to_excel(writer, sheet_name="top10_por_modelo", index=False)
    df_fallos.to_excel(writer, sheet_name="fallos", index=False)

print(f"Resultados guardados en: {NOMBRE_ARCHIVO_EXCEL}")
print(f"Filas resultados: {len(df)}")
print(f"Filas fallos: {len(df_fallos)}")


Resultados guardados en: ../resultados\resultados_modelos_alternativos_params_fijos.xlsx
Filas resultados: 170
Filas fallos: 0


In [9]:
# Vista rápida: mejores resultados por dataset y modelo
if not df.empty:
    display(
        df.sort_values(["dataset_logico", "nombre_modelo_aprendizaje", "test_f1_macro"], ascending=[True, True, False])[
            ["dataset_logico", "nombre_modelo_aprendizaje", "tipo_combination", "tecnica_aumento", "grado_limpieza",
             "test_f1_macro", "delta_f1_macro_vs_base_mismo_modelo", "test_avacc", "test_accuracy", "tiempo_total_seg"]
        ].head(40)
    )
else:
    print("Sin resultados.")


,dataset_logico,nombre_modelo_aprendizaje,tipo_combination,tecnica_aumento,grado_limpieza,test_f1_macro,delta_f1_macro_vs_base_mismo_modelo,test_avacc,test_accuracy,tiempo_total_seg
84,shuttle,XGBoost,contemporaneo,vs-smote,0,1.0000,0.0499,1.0000,1.0000,11.036
70,shuttle,XGBoost,contemporaneo,radius-smote,0,0.9956,0.0455,0.9995,0.9997,13.447
28,shuttle,XGBoost,clasico,borderlinesmote,0,0.9775,0.0274,0.9958,0.9999,9.041
42,shuttle,XGBoost,clasico,smote,0,0.9694,0.0193,0.9524,0.9999,10.743
56,shuttle,XGBoost,contemporaneo,ld-smote,0,0.9673,0.0172,0.9523,0.9998,10.893
1,shuttle,XGBoost,base,base,1,0.9520,0.0000,0.9281,0.9997,1.724
0,shuttle,XGBoost,base,base,0,0.9501,0.0000,0.9243,0.9997,1.678
14,shuttle,XGBoost,clasico,adasyn,0,0.9297,-0.0204,0.9244,0.9998,10.546
71,shuttle,XGBoost,contemporaneo,radius-smote,1,0.9215,-0.0305,0.9977,0.9959,13.231
43,shuttle,XGBoost,clasico,smote,1,0.8952,-0.0568,0.9510,0.9993,10.405
